In [ ]:
import os, glob, time
import numpy as np, pandas as pd
from numba import njit
from joblib import Parallel, delayed

def find_root():
    for r in ["/kaggle/input/rogii-wellbore-geology-prediction",
              "/kaggle/input/competitions/rogii-wellbore-geology-prediction","data"]+sorted(glob.glob("/kaggle/input/*")):
        if glob.glob(os.path.join(r,"train","*__horizontal_well.csv")): return r
    raise FileNotFoundError("data")
DATA=find_root(); print("DATA:",DATA,flush=True)

@njit(cache=False)
def viterbi(gr, dgr, twg, twdg, grid0, dstep, band, w_abs, w_grad, k_smooth, start_j):
    n=len(gr); G=len(twg)
    NEG=1e18
    cost=np.full(G,NEG); 
    for j in range(G): cost[j]=NEG
    cost[start_j]=0.0
    bptr=np.zeros((n,G),np.int32)
    curr=np.full(G,NEG)
    for i in range(n):
        for j in range(G):
            e = w_abs*abs(gr[i]-twg[j]) + w_grad*abs(dgr[i]-twdg[j])
            best=NEG; ba=j
            lo=j-band if j-band>0 else 0
            hi=j+band+1 if j+band+1<G else G
            for jp in range(lo,hi):
                v=cost[jp]+k_smooth*(j-jp)*(j-jp)
                if v<best: best=v; ba=jp
            curr[j]=best+e; bptr[i,j]=ba
        for j in range(G): cost[j]=curr[j]
    # backtrack from best final
    jb=0; bv=cost[0]
    for j in range(1,G):
        if cost[j]<bv: bv=cost[j]; jb=j
    path=np.zeros(n,np.int64)
    for i in range(n-1,-1,-1):
        path[i]=jb; jb=bptr[i,jb]
    return grid0+path*dstep

def predict(path, w):
    h=pd.read_csv(f"{DATA}/train/{w}__horizontal_well.csv",usecols=["MD","Z","GR","TVT","TVT_input"])
    tw=pd.read_csv(f"{DATA}/train/{w}__typewell.csv")
    ps=int(h["TVT_input"].notna().sum())
    if ps<60 or ps>=len(h)-5: return None
    tw_s=tw.dropna(subset=["TVT","GR"]).sort_values("TVT")
    if len(tw_s)<10: return None
    tps=float(h["TVT_input"].iloc[ps-1])
    gr=h["GR"].interpolate(limit_direction="both").fillna(tw_s["GR"].mean()).to_numpy(float)
    tvt=h["TVT"].to_numpy(float); z=h["Z"].to_numpy(float)
    # typewell grid
    band_ft=70.0; dstep=0.5
    lo=tps-band_ft; hi=tps+band_ft
    grid=np.arange(lo,hi,dstep)
    twg=np.interp(grid, tw_s["TVT"].to_numpy(), tw_s["GR"].to_numpy())
    twdg=np.gradient(twg)
    ev=np.arange(ps,len(h))
    ge=gr[ev]; dge=np.gradient(pd.Series(gr).rolling(9,center=True,min_periods=1).mean().to_numpy())[ev]
    gsig=max(np.nanstd(gr[:ps]-np.interp(h["TVT_input"].to_numpy(float)[:ps],grid,twg)),8.0)
    start_j=int((tps-lo)/dstep)
    trans=int(6.0/dstep)   # max ~6 ft dip per step search band
    out=viterbi(ge.astype(np.float64), dge.astype(np.float64), twg.astype(np.float64), twdg.astype(np.float64),
                float(lo), float(dstep), trans, 1.0/gsig, 0.6/gsig, 0.02, start_j)
    y=tvt[ev]
    return (y, out, np.full(len(y),tps))

wells=[os.path.basename(f).split("__")[0] for f in sorted(glob.glob(f"{DATA}/train/*__horizontal_well.csv"))]
print("wells",len(wells),flush=True)
t0=time.time()
res=Parallel(n_jobs=4,verbose=10)(delayed(predict)(None,w) for w in wells)
res=[r for r in res if r is not None]
def pooled(key):
    e=[(y-(p if key=="dp" else c))**2 for y,p,c in res]
    return float(np.sqrt(np.mean(np.concatenate(e))))
print("wells used",len(res),"%.0fs"%(time.time()-t0),flush=True)
print("const pooled %.3f"%pooled("const"),flush=True)
print("DP-align pooled %.3f"%pooled("dp"),flush=True)
# test-like subset
try:
    hf=glob.glob("/kaggle/input/rogii-cv-harness/well_table_with_dist.csv")
    if hf:
        wt=pd.read_csv(hf[0]); near=set(wt.loc[wt["dist_test"]<10000,"well"])
        idx=[i for i,w in enumerate([w for w in wells]) if w in near]
        # rebuild pairs aligned to res order is messy; recompute mask by re-run wells order
    print("DONE",flush=True)
except Exception as e: print("harness",e)